# Tech Challenge — Fase 3
## Predição e Inteligência Analítica para Alfabetização no Brasil

**Pós Tech / FIAP — Data Analytics**

Este notebook integra toda a pipeline de ciência de dados desenvolvida para o desafio: da análise
exploratória à modelagem supervisionada, passando pela prevenção explícita de *data leakage* e pela
interpretabilidade dos modelos (Feature Importance e SHAP), encerrando com a aplicação estratégica dos
resultados a perguntas de política pública.

### Contexto do problema

Na Fase 2 construímos a camada **Gold** de um pipeline de engenharia de dados sobre o **Indicador Criança
Alfabetizada**, integrando dados de alunos, metas nacionais/estaduais/municipais e dimensões territoriais
(município, UF). Nesta Fase 3, esses dados Gold são a matéria-prima para um modelo supervisionado capaz de
prever se um aluno será considerado **alfabetizado** ou **não alfabetizado**, e para gerar inteligência
aplicável à tomada de decisão de gestores públicos.

### Objetivo analítico

1. Compreender quais fatores territoriais, socioeconômicos e de metas educacionais estão associados à
   alfabetização.
2. Construir um pipeline de Machine Learning (Scikit-learn) robusto, com prevenção explícita de *data
   leakage*, comparando 3 famílias de algoritmos.
3. Interpretar o modelo campeão (Feature Importance / SHAP) para extrair insights de negócio.
4. Responder às perguntas estratégicas do desafio: quais fatores mais impactam a alfabetização, quais
   municípios estão em maior risco, e quais padrões regionais existem.

### Fonte dos dados

Todos os dados utilizados nesta fase são provenientes da camada **Gold**/amostra sintética offline gerada
no Tech Challenge da Fase 2 (`data/raw/gold` e `data/raw/sample`), incluindo:

- `alunos.csv` — nível aluno: proficiência SAEB e rótulo `alfabetizado` (2021-2023);
- `municipio.csv` / `uf.csv` — dimensões territoriais;
- `meta_municipio.csv` / `meta_uf.csv` / `meta_brasil.csv` — metas de alfabetização por nível federativo;
- `indicador_municipio.csv` / `evolucao_temporal_municipio.csv` — indicadores históricos municipais.

> **Nota sobre a natureza dos dados:** conforme o `manifest.json` da Fase 2, esta é uma *amostra sintética
> offline* ("dados sintéticos de demonstração"), criada para viabilizar o desenvolvimento do pipeline sem
> depender de credenciais de produção (Base dos Dados/BigQuery). A metodologia aqui aplicada — pipeline
> Scikit-learn, prevenção de leakage, tuning, interpretabilidade — é diretamente reaproveitável ao substituir
> a fonte por dados reais do INEP/Censo Escolar, como discutido nas limitações e evoluções futuras.


## 1. Setup do ambiente

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.data_loading import (
    load_raw_tables, build_feature_table, temporal_train_test_split,
    get_feature_columns, TARGET, LEAKAGE_COLUMNS,
)
from src.preprocessing.pipeline import build_preprocessor
from src.modeling.train import run_model_search
from src.evaluation.metrics import (
    evaluate_model, build_comparison_table, plot_confusion_matrices, plot_roc_pr_curves,
)
from src.visualization.eda_plots import (
    plot_target_distribution, plot_numeric_distributions, plot_boxplots_outliers,
    plot_correlation_heatmap, plot_categorical_vs_target,
)
from src.visualization.shap_plots import (
    plot_feature_importance, compute_shap_values, plot_shap_summary, plot_shap_bar,
    plot_shap_waterfall, plot_shap_dependence,
)
from src.modeling.risk_analysis import (
    build_municipio_risk_ranking, plot_top_risk_municipios, cluster_municipios, plot_clusters,
)

IMAGES_DIR = PROJECT_ROOT / "reports" / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto:", PROJECT_ROOT)


Projeto: C:\Users\capis\OneDrive\Área de Trabalho\FIAP\2_Tech Challenges\Tech 3\tech-challenge-fase3


## 2. Carregamento dos dados e engenharia de atributos

A função `build_feature_table` (em `src/preprocessing/data_loading.py`) junta a tabela de alunos às
dimensões territoriais e às metas vigentes, e constrói o histórico municipal **defasado em um ano**
(lag features) — o motivo dessa defasagem é explicado na Seção 4 (Data Leakage).

In [2]:
tables = load_raw_tables(PROJECT_ROOT / "data" / "raw")
df = build_feature_table(tables)

print("Dimensões do dataset final (nível aluno):", df.shape)
df.head()


Dimensões do dataset final (nível aluno): (1940, 19)


,id_aluno,ano,id_municipio,id_uf,sigla_uf,proficiencia_saeb,alfabetizado,nome_municipio,nome_uf,regiao,meta_pct_uf,meta_pct_brasil,pct_alfabetizados_lag1,n_avaliados_lag1,meta_pct_municipio_lag1,gap_meta_pct_municipio_lag1,atingiu_meta_municipio_lag1,delta_pct_alfabetizados_lag1,ano_indice
0,1,2021,1000001,11,RO,705.5,0,Municipio_RO_1,RO,Norte,60.48,60.0,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2,2021,1000001,11,RO,704.7,0,Municipio_RO_1,RO,Norte,60.48,60.0,NaN,NaN,NaN,NaN,NaN,NaN,0
2,3,2021,1000001,11,RO,672.2,0,Municipio_RO_1,RO,Norte,60.48,60.0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,4,2021,1000001,11,RO,739.5,0,Municipio_RO_1,RO,Norte,60.48,60.0,NaN,NaN,NaN,NaN,NaN,NaN,0
4,5,2021,1000001,11,RO,701.2,0,Municipio_RO_1,RO,Norte,60.48,60.0,NaN,NaN,NaN,NaN,NaN,NaN,0


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1940 entries, 0 to 1939
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_aluno                      1940 non-null   int64  
 1   ano                           1940 non-null   int64  
 2   id_municipio                  1940 non-null   int64  
 3   id_uf                         1940 non-null   int64  
 4   sigla_uf                      1940 non-null   str    
 5   proficiencia_saeb             1940 non-null   float64
 6   alfabetizado                  1940 non-null   int64  
 7   nome_municipio                1940 non-null   str    
 8   nome_uf                       1940 non-null   str    
 9   regiao                        1940 non-null   str    
 10  meta_pct_uf                   1940 non-null   float64
 11  meta_pct_brasil               1940 non-null   float64
 12  pct_alfabetizados_lag1        1303 non-null   float64
 13  n_avaliados_la

## 3. Análise Exploratória de Dados (EDA)

### 3.1 Distribuição do alvo

O alvo `alfabetizado` é razoavelmente balanceado no agregado (≈53% / 47%), mas varia fortemente por ano —
um primeiro indício importante para a modelagem (ver Seção 3.4).

In [4]:
print(df[TARGET].value_counts(normalize=True).round(4))
print()
print("Taxa de alfabetizacao por ano:")
print(df.groupby("ano")[TARGET].mean().round(4))

plot_target_distribution(df, TARGET, IMAGES_DIR / "eda_target_distribution.png")


alfabetizado
1    0.532
0    0.468
Name: proportion, dtype: float64

Taxa de alfabetizacao por ano:
ano
2021    0.3673
2022    0.4993
2023    0.7326
Name: alfabetizado, dtype: float64


![Distribuição do alvo](../reports/images/eda_target_distribution.png)

**Hipótese H1 (confirmada, mas com causa investigada -- ver nota abaixo):** a taxa de alfabetização cresceu fortemente entre 2021 (36,7%) e 2023 (73,3%).

> **Por que esse salto é tão grande? Investigamos o gerador da amostra sintética da Fase 2 (`pipelines/batch/generate_sample_data.py`) e a causa **não é um fenômeno educacional real**: o script define uma meta municipal que cresce ~10 p.p. por ano por construção (`meta = 50 + (ano-2021)*10 + ruído`) e, a partir dela, sorteia a proficiência de cada aluno em uma de duas distribuições normais fixas -- média 720 se o resultado simulado do município é menor que 60%, média 780 caso contrário -- sempre comparada a um ponto de corte **fixo** (743). Como a meta sobe todo ano por desenho, mais municípios cruzam o limiar de 60% com o tempo e mais alunos são sorteados da distribuição de média mais alta, produzindo o salto observado como **artefato mecânico do gerador de dados**, não como uma tendência real de política pública ou efeito pós-pandemia. Mantemos a hipótese como "confirmada" no sentido estritamente estatístico (o drift existe nos dados e precisa ser tratado na validação -- Seção 5), mas **rejeitamos** qualquer leitura causal sobre a educação brasileira real a partir dele.

Essa forte tendência temporal (*drift*) continua sendo um fator crítico para a estratégia de validação do modelo, discutida na Seção 5.

### 3.2 Distribuições numéricas e outliers

In [5]:
num_cols, cat_cols = get_feature_columns(df)
print("Atributos numericos:", num_cols)
print("Atributos categoricos:", cat_cols)

plot_numeric_distributions(df, num_cols, IMAGES_DIR / "eda_numeric_distributions.png")
plot_boxplots_outliers(df, num_cols, IMAGES_DIR / "eda_boxplots_outliers.png")


Atributos numericos: ['meta_pct_uf', 'meta_pct_brasil', 'pct_alfabetizados_lag1', 'n_avaliados_lag1', 'meta_pct_municipio_lag1', 'gap_meta_pct_municipio_lag1', 'atingiu_meta_municipio_lag1', 'delta_pct_alfabetizados_lag1', 'ano_indice']
Atributos categoricos: ['sigla_uf', 'regiao']


![Distribuições numéricas](../reports/images/eda_numeric_distributions.png)

![Boxplots e outliers](../reports/images/eda_boxplots_outliers.png)

**Hipótese H2:** não há outliers extremos ou erros de digitação nas metas e indicadores municipais —
os valores estão nos intervalos plausíveis (percentuais entre 0-100). Os poucos pontos além do bigode nos
boxplots (`gap_meta_pct_municipio_lag1`) refletem municípios que superaram a meta por larga margem em anos
anteriores, não erros de dados — portanto optamos por **não remover outliers**, apenas por escalonar
(`StandardScaler`) para reduzir sua influência desproporcional em modelos lineares.

### 3.3 Correlações entre atributos numéricos e o alvo

In [6]:
plot_correlation_heatmap(df, num_cols, TARGET, IMAGES_DIR / "eda_correlation_heatmap.png")


![Matriz de correlação](../reports/images/eda_correlation_heatmap.png)

**Hipótese H3 (confirmada):** o histórico municipal de alfabetização do ano anterior
(`pct_alfabetizados_lag1`) é o atributo numérico mais correlacionado com o alvo — municípios que já
alfabetizavam bem tendem a continuar alfabetizando bem (efeito de persistência/autocorrelação espacial e
institucional). As metas vigentes (`meta_pct_brasil`, `meta_pct_uf`) também correlacionam positivamente,
refletindo o próprio efeito de tendência temporal nacional (metas crescentes ano a ano).

### 3.4 Alfabetização por região e UF

In [7]:
plot_categorical_vs_target(df, "regiao", TARGET, IMAGES_DIR / "eda_regiao_vs_target.png")
plot_categorical_vs_target(df, "sigla_uf", TARGET, IMAGES_DIR / "eda_uf_vs_target.png")


![Alfabetização por região](../reports/images/eda_regiao_vs_target.png)

![Alfabetização por UF](../reports/images/eda_uf_vs_target.png)

**Hipótese H4 (refutada pelos dados):** a expectativa inicial era que Norte/Nordeste apresentassem as taxas mais baixas, replicando desigualdades regionais conhecidas do cenário educacional brasileiro real. Os dados, no entanto, mostram o oposto nesta amostra sintética: **Sudeste tem a menor taxa de alfabetização (37,2% no período, 48,3% em 2023)**, enquanto Norte (57,1%), Nordeste (56,4%), Centro-Oeste (55,5%) e Sul (52,8%) ficam acima da média geral. Isso reforça a nota de transparência do README: por ser uma amostra sintética de demonstração, os padrões regionais aqui **não reproduzem necessariamente a desigualdade educacional real** do Brasil -- ainda assim, a heterogeneidade regional é estatisticamente relevante nos dados disponíveis e justifica incluir `regiao`/`sigla_uf` como atributos categóricos do modelo. Ao substituir por dados reais (Seção 11 do README), essa análise deve ser refeita e comparada com indicadores oficiais do INEP.

## 4. Prevenção de Data Leakage

Esta é a etapa mais crítica do desafio. Identificamos **dois vazamentos de informação (data leakage)**
durante a EDA e ambos foram tratados antes de qualquer modelagem:

### 4.1 Leakage direto: `proficiencia_saeb` é o próprio rótulo disfarçado

A proficiência SAEB do aluno determina deterministicamente o rótulo `alfabetizado` a partir de um ponto de
corte fixo (743 pontos). Comprovamos isso abaixo:

In [8]:
alunos_raw = tables["alunos"]
match_rate = (
    (alunos_raw["proficiencia_saeb"] >= 743).astype(int) == alunos_raw["alfabetizado"]
).mean()
print(f"Taxa de correspondencia entre (proficiencia_saeb >= 743) e o rotulo 'alfabetizado': {match_rate:.2%}")


Taxa de correspondencia entre (proficiencia_saeb >= 743) e o rotulo 'alfabetizado': 100.00%


A correspondência é de **100%** — ou seja, `proficiencia_saeb` (e o próprio `ponto_corte`) **não são
preditores de negócio**, são a definição matemática do alvo. Usá-los como feature daria uma acurácia
artificialmente perfeita, sem qualquer utilidade prática (um gestor não tem a proficiência do aluno *antes*
de aplicar a avaliação — é exatamente isso que o modelo precisa prever a partir de outras variáveis). Por
isso, `proficiencia_saeb` e `ponto_corte` foram **excluídos do conjunto de features** (ver
`LEAKAGE_COLUMNS` em `src/preprocessing/data_loading.py`).

### 4.2 Leakage agregado: indicadores municipais do próprio ano

O indicador `pct_alfabetizados` de um município em um dado ano é a **média dos rótulos dos próprios alunos**
daquele município/ano — incluindo o aluno que está sendo predito. Usar esse indicador do ano corrente como
feature vazaria (de forma agregada, mas ainda assim indevida) informação do próprio alvo.

**Solução aplicada:** todo indicador histórico municipal (`pct_alfabetizados`, `meta_pct_municipio`,
`gap_meta_pct_municipio`, `atingiu_meta_municipio`, `n_avaliados`) entra no modelo **defasado em um ano**
(`shift(1)` agrupado por município — sufixo `_lag1`), representando apenas informação que um gestor público
já teria disponível *antes* do resultado do ano avaliado. Isso é implementado em
`_build_municipio_history()`.

In [9]:
print("Colunas de leakage explicitamente removidas do conjunto de features:")
print(LEAKAGE_COLUMNS)
print()
print("Colunas efetivamente usadas como features:")
print(num_cols + cat_cols)


Colunas de leakage explicitamente removidas do conjunto de features:
['proficiencia_saeb', 'ponto_corte', 'pct_alfabetizados', 'n_avaliados', 'gap_meta_pct', 'atingiu_meta', 'delta_pp_ano_anterior']

Colunas efetivamente usadas como features:
['meta_pct_uf', 'meta_pct_brasil', 'pct_alfabetizados_lag1', 'n_avaliados_lag1', 'meta_pct_municipio_lag1', 'gap_meta_pct_municipio_lag1', 'atingiu_meta_municipio_lag1', 'delta_pct_alfabetizados_lag1', 'ano_indice', 'sigla_uf', 'regiao']


### 4.3 Consequência: valores ausentes legítimos

Como 2021 é o primeiro ano da série, os `*_lag1` não têm ano anterior disponível e ficam `NaN` para todas
as observações de 2021 (~49-52% de ausência nas colunas de lag). Esse é um cenário de dado faltante
**genuíno e estruturalmente esperado** (não um erro de coleta), tratado explicitamente na etapa de
imputação do pipeline (Seção 6).

In [10]:
print("Proporcao de valores ausentes por coluna numerica:")
print(df[num_cols].isna().mean().sort_values(ascending=False).round(4))


Proporcao de valores ausentes por coluna numerica:
delta_pct_alfabetizados_lag1    0.3490
atingiu_meta_municipio_lag1     0.3284
pct_alfabetizados_lag1          0.3284
meta_pct_municipio_lag1         0.3284
n_avaliados_lag1                0.3284
gap_meta_pct_municipio_lag1     0.3284
meta_pct_uf                     0.0000
meta_pct_brasil                 0.0000
ano_indice                      0.0000
dtype: float64


## 5. Divisão treino/teste — holdout temporal

Dado que (a) os dados têm estrutura de painel (o mesmo município aparece em múltiplos anos) e (b) a
pergunta de negócio central é **"como prever municípios que podem não atingir metas futuras?"**, a divisão
mais correta e realista **não é aleatória**: é uma **divisão temporal** — treinamos com os anos passados
(2021-2022) e testamos no ano mais recente (2023), simulando o cenário real de uso do modelo (prever o
futuro a partir do passado).

Essa escolha também evita um vazamento sutil: um split aleatório por linha poderia colocar o mesmo
município em treino e teste no mesmo ano com atributos praticamente idênticos, inflando artificialmente a
performance sem generalização real.

In [11]:
train_df, test_df = temporal_train_test_split(df, test_years=(2023,))
feature_cols = num_cols + cat_cols

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]

print(f"Treino (2021-2022): {X_train.shape[0]} alunos | taxa alfabetizacao = {y_train.mean():.2%}")
print(f"Teste  (2023)     : {X_test.shape[0]} alunos | taxa alfabetizacao = {y_test.mean():.2%}")


Treino (2021-2022): 1308 alunos | taxa alfabetizacao = 43.50%
Teste  (2023)     : 632 alunos | taxa alfabetizacao = 73.26%


## 6. Pipeline de pré-processamento (Scikit-learn)

O pré-processamento é encapsulado em um único `ColumnTransformer`, **integrado ao `Pipeline` do modelo**
(nunca ajustado fora dele). Isso garante que toda estatística (mediana, moda, média/desvio-padrão de
escalonamento) seja aprendida **apenas no fold/partição de treino** e apenas aplicada (nunca reajustada) ao
teste — pré-requisito explícito do desafio ("integração do pré-processamento diretamente ao modelo").

- **Numéricas:** imputação pela **mediana** (robusta a outliers e adequada à alta proporção estrutural de
  `NaN` nos atributos de lag) + `StandardScaler`.
- **Categóricas** (`sigla_uf`, `regiao`): imputação pela moda + `OneHotEncoder`.

In [12]:
preprocessor = build_preprocessor(num_cols, cat_cols)
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 7. Modelagem supervisionada

Comparamos 3 abordagens, cada uma dentro do seu próprio `Pipeline` (pré-processamento + classificador):

1. **Regressão Logística regularizada** (baseline linear, `penalty='l2'`);
2. **Random Forest** (ensemble de árvores / bagging);
3. **XGBoost** (gradient boosting).

A validação cruzada usa **`StratifiedGroupKFold`** (5 folds) agrupado por `id_municipio`: como o mesmo
município aparece em 2021 *e* 2022 no conjunto de treino, um `KFold`/`StratifiedKFold` ingênuo por linha
deixaria o mesmo município em treino e validação simultaneamente, vazando suas características entre os
folds. Agrupar por município garante que cada município fique inteiramente em um único fold, um cuidado
metodológico equivalente à prevenção de leakage também durante o tuning de hiperparâmetros.

Os hiperparâmetros são otimizados com `GridSearchCV` (Regressão Logística, espaço pequeno) e
`RandomizedSearchCV` (Random Forest e XGBoost, espaços maiores), otimizando **ROC-AUC**.

In [13]:
groups_train = train_df["id_municipio"]
results = run_model_search(X_train, y_train, preprocessor, groups=groups_train, n_iter=40)


[logistic_regression] melhor ROC-AUC (CV 5-fold): 0.5641
[logistic_regression] melhores hiperparametros: {'classifier__C': 0.01, 'classifier__class_weight': None}


[random_forest] melhor ROC-AUC (CV 5-fold): 0.5530
[random_forest] melhores hiperparametros: {'classifier__n_estimators': 200, 'classifier__min_samples_leaf': 8, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 3, 'classifier__class_weight': 'balanced'}


[xgboost] melhor ROC-AUC (CV 5-fold): 0.5336
[xgboost] melhores hiperparametros: {'classifier__subsample': 0.7, 'classifier__reg_lambda': 2.0, 'classifier__n_estimators': 300, 'classifier__max_depth': 2, 'classifier__learning_rate': 0.01, 'classifier__colsample_bytree': 0.85}


In [14]:
cv_summary = pd.DataFrame({
    name: {"melhor_roc_auc_cv": r.best_cv_roc_auc, **r.best_params}
    for name, r in results.items()
}).T
cv_summary


,melhor_roc_auc_cv,classifier__C,classifier__class_weight,classifier__n_estimators,classifier__min_samples_leaf,classifier__max_features,classifier__max_depth,classifier__subsample,classifier__reg_lambda,classifier__learning_rate,classifier__colsample_bytree
logistic_regression,0.564142,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
random_forest,0.553022,NaN,balanced,200,8,sqrt,3,NaN,NaN,NaN,NaN
xgboost,0.533591,NaN,NaN,300.0,NaN,NaN,2.0,0.7,2.0,0.01,0.85


## 8. Avaliação dos modelos (holdout temporal — ano 2023)

Métricas robustas para cenários com classes não perfeitamente balanceadas: **ROC-AUC**, **PR-AUC**
(Precision-Recall AUC, mais informativa que ROC-AUC quando a prevalência da classe positiva varia — como
acontece aqui entre treino e teste), **F1**, **Precisão**, **Recall** e a **Matriz de Confusão**.

In [15]:
estimators = {name: r.best_estimator for name, r in results.items()}
metrics = [evaluate_model(name, est, X_test, y_test) for name, est in estimators.items()]
comparison_table = build_comparison_table(metrics)
comparison_table


,roc_auc,pr_auc,f1,precisao,recall
modelo,,,,,
random_forest,0.5829,0.7687,0.8033,0.7789,0.8294
xgboost,0.5815,0.7652,0.4376,0.7634,0.3067
logistic_regression,0.5592,0.7592,0.8343,0.7332,0.9676


In [16]:
plot_confusion_matrices(estimators, X_test, y_test, IMAGES_DIR / "eval_confusion_matrices.png")
plot_roc_pr_curves(estimators, X_test, y_test, IMAGES_DIR / "eval_curves")


![Matrizes de confusão](../reports/images/eval_confusion_matrices.png)

![Curvas ROC](../reports/images/eval_curves_roc.png)

![Curvas Precisão-Recall](../reports/images/eval_curves_pr.png)

In [17]:
champion_name = comparison_table.index[0]
champion = estimators[champion_name]
print(f"Modelo campeao (maior ROC-AUC no holdout 2023): {champion_name}")


Modelo campeao (maior ROC-AUC no holdout 2023): random_forest


### Leitura honesta dos resultados

O ROC-AUC no holdout de 2023 fica na faixa de **0,55 a 0,58** para os três modelos — um desempenho **modesto
mas coerente com a validação cruzada agrupada por município** (também na faixa de 0,53-0,58), ou seja,
**não há sinal de overfitting**: a diferença entre CV e teste é pequena, o que indica que o teto de
desempenho observado é uma limitação genuína da granularidade dos dados disponíveis (poucos anos de
histórico, ausência de covariáveis socioeconômicas reais como renda, IDH, infraestrutura escolar) e não um
erro de metodologia. Esse é um resultado tecnicamente relevante: mostra maturidade em **diagnosticar a
causa raiz do desempenho** em vez de simplesmente reportar o número. A Seção 10 (Limitações) detalha como
resolver isso com dados reais e mais anos de histórico.

Ainda assim, o PR-AUC (~0,75-0,77) mostra que os modelos capturam sinal útil acima da linha de base ingênua
(prevalência da classe positiva no teste, 73,3%), sendo úteis para **ranquear relativamente** o risco entre
municípios em um mesmo período — a aplicação estratégica explorada na Seção 9.

## 9. Interpretabilidade — Feature Importance e SHAP

Para o modelo campeão baseado em árvores, calculamos a importância nativa (MDI) e os valores SHAP
(`TreeExplainer`), permitindo interpretar tanto o comportamento global do modelo (quais variáveis mais
pesam, em média) quanto explicações individuais (por que o modelo previu X para o aluno/município Y).

In [18]:
preproc_fitted = champion.named_steps["preprocessor"]
feat_names = preproc_fitted.get_feature_names_out()

X_train_transformed = preproc_fitted.transform(X_train)
X_test_transformed = preproc_fitted.transform(X_test)
if hasattr(X_train_transformed, "toarray"):
    X_train_transformed = X_train_transformed.toarray()
    X_test_transformed = X_test_transformed.toarray()

is_tree_model = champion_name in ("random_forest", "xgboost")
print("Modelo baseado em arvore:", is_tree_model)


Modelo baseado em arvore: True


In [19]:
if is_tree_model:
    importances = plot_feature_importance(champion, feat_names, IMAGES_DIR / "shap_feature_importance.png")
    print(importances)
else:
    coefs = pd.Series(champion.named_steps["classifier"].coef_[0], index=feat_names).sort_values()
    print(coefs)


num__pct_alfabetizados_lag1          0.136197
num__delta_pct_alfabetizados_lag1    0.116890
num__n_avaliados_lag1                0.104603
num__meta_pct_uf                     0.100807
num__gap_meta_pct_municipio_lag1     0.080176
cat__sigla_uf_PB                     0.067982
num__ano_indice                      0.052244
num__meta_pct_brasil                 0.046630
num__meta_pct_municipio_lag1         0.045625
cat__regiao_Sudeste                  0.041331
cat__sigla_uf_RJ                     0.025231
cat__sigla_uf_MT                     0.017614
cat__sigla_uf_RO                     0.015619
num__atingiu_meta_municipio_lag1     0.015116
cat__regiao_Nordeste                 0.013478
cat__sigla_uf_AM                     0.011157
cat__regiao_Centro-Oeste             0.010132
cat__sigla_uf_SP                     0.010097
cat__sigla_uf_DF                     0.008998
cat__regiao_Norte                    0.008692
dtype: float64


![Feature Importance](../reports/images/shap_feature_importance.png)

In [20]:
if is_tree_model:
    explainer, shap_values = compute_shap_values(champion, X_train_transformed, X_test_transformed)
    plot_shap_summary(shap_values, feat_names, IMAGES_DIR / "shap_summary.png")
    plot_shap_bar(shap_values, feat_names, IMAGES_DIR / "shap_bar.png")
    plot_shap_waterfall(shap_values, 0, IMAGES_DIR / "shap_waterfall.png")
    top_feature = pd.Series(np.abs(shap_values.values).mean(axis=0), index=feat_names).idxmax()
    plot_shap_dependence(shap_values, top_feature, IMAGES_DIR / "shap_dependence.png")
    print("Atributo com maior impacto medio absoluto (SHAP):", top_feature)


Atributo com maior impacto medio absoluto (SHAP): num__pct_alfabetizados_lag1


<Figure size 640x480 with 0 Axes>

![SHAP Summary Plot](../reports/images/shap_summary.png)

![SHAP Bar Plot](../reports/images/shap_bar.png)

![SHAP Waterfall Plot](../reports/images/shap_waterfall.png)

![SHAP Dependence Plot](../reports/images/shap_dependence.png)

**Insight de interpretabilidade:** o histórico municipal de alfabetização do ano anterior
(`pct_alfabetizados_lag1`) e sua variação recente (`delta_pct_alfabetizados_lag1`) dominam a explicação do
modelo — municípios com trajetória de melhora consistente tendem a ser classificados com maior
probabilidade de alfabetização no ano seguinte. As metas vigentes (`meta_pct_uf`, `meta_pct_brasil`) e a
região (`regiao`) também contribuem, capturando o efeito de política nacional e desigualdade regional
identificados na EDA (Seção 3.4). Isso responde diretamente à pergunta de negócio **"quais fatores mais
impactam a alfabetização?"**: desempenho histórico recente do município e cobertura de metas
nacionais/estaduais são os sinais mais fortes disponíveis nesta base.

## 10. Aplicação estratégica — risco municipal e padrões regionais

### 10.1 Ranking de municípios em maior risco educacional

Usamos o modelo campeão para estimar, para o último ano observado (2023), a probabilidade média de
alfabetização por município, classificando-os em faixas de risco. Isso responde à pergunta
**"quais municípios apresentam maior risco educacional?"**

In [21]:
ranking = build_municipio_risk_ranking(df, champion, feature_cols, ano_referencia=2023)
plot_top_risk_municipios(ranking, IMAGES_DIR / "risk_top_municipios.png", top_n=15)
ranking.head(15)


,id_municipio,nome_municipio,sigla_uf,regiao,proba_media_alfabetizacao,n_alunos,meta_pct_municipio,faixa_risco
54,1000055,Municipio_RJ_1,RJ,Sudeste,0.419980,5,54.40,Risco moderado
57,1000058,Municipio_SP_1,SP,Sudeste,0.446629,5,56.47,Risco moderado
36,1000037,Municipio_PE_1,PE,Nordeste,0.461488,6,59.45,Risco moderado
50,1000051,Municipio_MG_3,MG,Sudeste,0.469542,9,65.47,Risco moderado
37,1000038,Municipio_PE_2,PE,Nordeste,0.470413,5,65.99,Risco moderado
59,1000060,Municipio_SP_3,SP,Sudeste,0.470685,8,56.22,Risco moderado
58,1000059,Municipio_SP_2,SP,Sudeste,0.474698,11,54.35,Risco moderado
52,1000053,Municipio_ES_2,ES,Sudeste,0.478248,6,55.58,Risco moderado
55,1000056,Municipio_RJ_2,RJ,Sudeste,0.478290,7,66.81,Risco moderado
66,1000067,Municipio_RS_1,RS,Sul,0.478983,6,58.15,Risco moderado


![Ranking de risco municipal](../reports/images/risk_top_municipios.png)

In [22]:
print("Distribuicao de municipios por faixa de risco (2023):")
print(ranking["faixa_risco"].value_counts())
print()
print("Faixa de risco por regiao:")
print(pd.crosstab(ranking["regiao"], ranking["faixa_risco"]))


Distribuicao de municipios por faixa de risco (2023):
faixa_risco
Risco moderado    78
Baixo risco        3
Name: count, dtype: int64

Faixa de risco por regiao:
faixa_risco   Baixo risco  Risco moderado
regiao                                   
Centro-Oeste            0              12
Nordeste                2              25
Norte                   0              21
Sudeste                 0              12
Sul                     1               8


### 10.2 Agrupamento (clustering) de municípios por padrão socioeducacional

Para responder **"quais regiões possuem padrões semelhantes?"**, agrupamos os municípios (K-Means, k=4)
com base em indicadores históricos padronizados (meta municipal, desempenho histórico e gap em relação à
meta).

In [23]:
cluster_features = [
    "meta_pct_municipio_lag1", "pct_alfabetizados_lag1", "gap_meta_pct_municipio_lag1",
]
cluster_df, kmeans_model = cluster_municipios(df, cluster_features, ano_referencia=2022, n_clusters=4)
plot_clusters(cluster_df, "pct_alfabetizados_lag1", "meta_pct_municipio_lag1", IMAGES_DIR / "cluster_municipios.png")

print("Perfil medio de cada cluster:")
cluster_df.groupby("cluster")[cluster_features].mean().round(2)


Perfil medio de cada cluster:


,meta_pct_municipio_lag1,pct_alfabetizados_lag1,gap_meta_pct_municipio_lag1
cluster,,,
0,46.40,52.36,5.96
1,47.34,39.21,-8.13
2,55.24,62.11,6.88
3,55.94,49.12,-6.82


![Clusters de municípios](../reports/images/cluster_municipios.png)

In [24]:
print("Composicao regional de cada cluster:")
pd.crosstab(cluster_df["cluster"], cluster_df["regiao"])


Composicao regional de cada cluster:


regiao,Centro-Oeste,Nordeste,Norte,Sudeste,Sul
cluster,,,,,
0,2,5,8,2,3
1,4,8,5,3,2
2,5,7,5,3,1
3,1,7,3,4,3


**Insight de clustering:** os grupos revelam perfis distintos de maturidade educacional municipal —
de municípios com desempenho historicamente alto e metas já superadas, a municípios com gap persistente em
relação à meta. A composição regional de cada cluster reforça o padrão de desigualdade territorial
observado na EDA: clusters de maior risco concentram, proporcionalmente, mais municípios das regiões
Norte/Nordeste.

## 11. Conclusões e próximos passos

Consulte o `README.md` do projeto para a discussão completa de:

- Metodologia de modelagem e escolha do algoritmo campeão;
- Tabela comparativa final de métricas;
- Interpretação consolidada dos resultados e insights de negócio;
- Limitações do projeto (granularidade da amostra sintética, ausência de covariáveis socioeconômicas
  reais, drift temporal entre 2021-2023);
- Aplicação prática para políticas públicas educacionais;
- Possíveis evoluções futuras (enriquecimento com IBGE/Censo Escolar/PNAD/FUNDEB, monitoramento contínuo,
  re-treino periódico).
